In [4]:
import warnings
warnings.filterwarnings('ignore')

In [5]:
!pip install graphframes-py==0.11.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 1.9 MB/s eta 0:00:00


In [6]:
import pyspark
pyspark.__version__

'4.0.3'

In [7]:
from pyspark.sql import SparkSession
from graphframes import GraphFrame
from pyspark.sql.functions import col, count


In [8]:
spark = SparkSession.builder.config("spark.jars.packages", "io.graphframes:graphframes-spark4_2.13:0.11.0").getOrCreate()

### Read departuredelays.csv in Edge DataFrame
### Read airport-codes-na.txt in Vertix DataFrame (the separator is Tab i.e sep = '\t' )

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [11]:
edges = spark.read.csv("/content/drive/MyDrive/departuredelays.csv", header=True, inferSchema=True)
edges.printSchema()


root
 |-- date: integer (nullable = true)
 |-- delay: integer (nullable = true)
 |-- distance: integer (nullable = true)
 |-- origin: string (nullable = true)
 |-- destination: string (nullable = true)



In [12]:
vertices = spark.read.csv(
    "/content/drive/MyDrive/airport-codes-na.txt",
    header=True,
    sep="\t",
    inferSchema=True
)
vertices.printSchema()

root
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- IATA: string (nullable = true)



#### The US flight delays data set has five columns:
- The <b>date</b> column contains an integer like 02190925 . When converted, this maps to 02-19 09:25 am.
- The <b>delay</b> column gives the delay in minutes between the scheduled and actual departure times. Early departures show negative numbers.
- The <b>distance</b> column gives the distance in miles from the origin airport to the destination airport.
- The <b>origin</b> column contains the origin IATA airport code.
- The <b>destination</b> column contains the destination IATA airport code.

#### The airport-codes data set has four columns:
- The <b>IATA</b> column contains IATA airport code.
- The <b>City, State, and Country</b> columns contains information about the airport location.

### In the vertix DataFrame, drop any duplicated rows with the same  IATA code.

In [13]:
vertices_duplicates = (vertices.groupBy("IATA").count().filter(col("count") > 1))
vertices_duplicates.show()

+----+-----+
|IATA|count|
+----+-----+
| Big|    3|
+----+-----+



In [14]:
vertices = vertices.dropDuplicates(["IATA"])


In [15]:
duplicates = (vertices.groupBy("IATA").count().filter(col("count") > 1))
duplicates.show()

+----+-----+
|IATA|count|
+----+-----+
+----+-----+



### In the edges DataFrame:
- Rename the <b>date</b> columns to become <b>tripid</b>.
- Rename the <b>origin</b> columns to become <b>src</b>.
- Rename the <b>destination</b> columns to become <b>dst</b>.

In [16]:
edges = edges.withColumnRenamed("date", "tripid").withColumnRenamed("origin", "src").withColumnRenamed("destination", "dst")

### In the Vertix DataFrame:
- Rename the <b>IATA</b> columns to become <b>id</b>.

In [17]:
vertices = vertices.withColumnRenamed("IATA", "id")


In [18]:
edges.show(5)


+-------+-----+--------+---+---+
| tripid|delay|distance|src|dst|
+-------+-----+--------+---+---+
|1011245|    6|     602|ABE|ATL|
|1020600|   -8|     369|ABE|DTW|
|1021245|   -2|     602|ABE|ATL|
|1020605|   -4|     602|ABE|ATL|
|1031245|   -4|     602|ABE|ATL|
+-------+-----+--------+---+---+
only showing top 5 rows


In [19]:
vertices.show(5)

+-----------+-----+-------+---+
|       City|State|Country| id|
+-----------+-----+-------+---+
|  Allentown|   PA|    USA|ABE|
|    Abilene|   TX|    USA|ABI|
|Albuquerque|   NM|    USA|ABQ|
|   Aberdeen|   SD|    USA|ABR|
|     Albany|   GA|    USA|ABY|
+-----------+-----+-------+---+
only showing top 5 rows


### Create GraphFrame from Vertix and Edges DataFrames

In [20]:
g = GraphFrame(vertices, edges)
g.vertices.show()
g.edges.show()

+-------------+-----+-------+---+
|         City|State|Country| id|
+-------------+-----+-------+---+
|    Allentown|   PA|    USA|ABE|
|      Abilene|   TX|    USA|ABI|
|  Albuquerque|   NM|    USA|ABQ|
|     Aberdeen|   SD|    USA|ABR|
|       Albany|   GA|    USA|ABY|
|    Nantucket|   MA|    USA|ACK|
|         Waco|   TX|    USA|ACT|
|       Eureka|   CA|    USA|ACV|
|Atlantic City|   NJ|    USA|ACY|
|       Kodiak|   AK|    USA|ADQ|
|   Alexandria|   LA|    USA|AEX|
|      Augusta|   GA|    USA|AGS|
|       Athens|   GA|    USA|AHN|
|     Alliance|   NE|    USA|AIA|
|  King Salmon|   AK|    USA|AKN|
|       Albany|   NY|    USA|ALB|
|     Waterloo|   IA|    USA|ALO|
|      Alamosa|   CO|    USA|ALS|
|  Walla Walla|   WA|    USA|ALW|
|     Amarillo|   TX|    USA|AMA|
+-------------+-----+-------+---+
only showing top 20 rows
+-------+-----+--------+---+---+
| tripid|delay|distance|src|dst|
+-------+-----+--------+---+---+
|1011245|    6|     602|ABE|ATL|
|1020600|   -8|     369|ABE

### Determine the number of airports

In [21]:
print("number of airports: ", g.vertices.count())

number of airports:  524


### Determine the number of trips

In [22]:
print("number of trips: ", g.edges.count())

number of trips:  1391578


### What is the longest delay?

In [23]:
print(f"the longest delay is {g.edges.orderBy(col('delay').desc()).first()['delay']}")

the longest delay is 1642


### Find out the number of delayed flights vs. early flights (flights that departed before actual time)

In [24]:
print("number of delayed trips: ", g.edges.filter(col("delay") > 0).count())
print("number of early trips: ", g.edges.filter(col("delay") < 0).count())

number of delayed trips:  591727
number of early trips:  668729


### What flight destinations departing SFO are most likely to have significant delays? Select the top 10
#### Hint: you should get the average delay for each destination for trips that depart from SFO only

In [25]:
from pyspark.sql.functions import avg

g.edges.filter("src = 'SFO'") \
       .groupBy("dst") \
       .agg(avg("delay").alias("avg_delay")) \
       .orderBy("avg_delay", ascending=False) \
       .show(10)

+---+------------------+
|dst|         avg_delay|
+---+------------------+
|JAC| 30.78846153846154|
|OKC|24.822222222222223|
|SUN|22.696629213483146|
|COS| 22.58888888888889|
|SAT|             22.16|
|STL|         20.203125|
|HNL|19.982608695652175|
|ASE|19.846153846153847|
|CEC|19.089820359281436|
|MDW|18.771929824561404|
+---+------------------+
only showing top 10 rows


### Find the Incoming connections to the airport sorted in Desc. order.

In [26]:
g.inDegrees.orderBy("inDegree", ascending=False).show()

+---+--------+
| id|inDegree|
+---+--------+
|ATL|   90434|
|DFW|   66050|
|ORD|   61967|
|LAX|   53601|
|DEN|   50921|
|IAH|   42700|
|PHX|   39721|
|SFO|   38988|
|LAS|   32994|
|CLT|   28388|
|MCO|   27959|
|EWR|   27652|
|LGA|   25469|
|BOS|   25360|
|SLC|   25323|
|JFK|   23484|
|DTW|   23310|
|SEA|   23074|
|MSP|   22385|
|MIA|   21805|
+---+--------+
only showing top 20 rows


### Find the Outgoing connections from the airport sorted in Desc. order.

In [27]:
g.outDegrees.orderBy("outDegree", ascending=False).show()

+---+---------+
| id|outDegree|
+---+---------+
|ATL|    91484|
|DFW|    68482|
|ORD|    64228|
|LAX|    54086|
|DEN|    53148|
|IAH|    43361|
|PHX|    40155|
|SFO|    39483|
|LAS|    33107|
|CLT|    28402|
|MCO|    28313|
|EWR|    27656|
|SLC|    25868|
|LGA|    25458|
|BOS|    25348|
|MSP|    24031|
|JFK|    23572|
|DTW|    23421|
|SEA|    23078|
|MIA|    21817|
+---+---------+
only showing top 20 rows


### Use motif finding to answer this question: which delays could we blame on SFO?
#### Hint: this practically means that SFO is a transit station

In [28]:
motifs = g.find("(a)-[ab]->(b); (b)-[bc]->(c)")

In [29]:
motifs.filter("b.id = 'SFO' and ab.delay > 0 and bc.delay > 0").select(
    "a.id",
    "b.id",
    "c.id",
    "ab.delay",
    "bc.delay"
).show()

+---+---+---+-----+-----+
| id| id| id|delay|delay|
+---+---+---+-----+-----+
|ABQ|SFO|JFK|    4|   55|
|ABQ|SFO|DFW|    4|  134|
|ABQ|SFO|ORD|    4|   32|
|ABQ|SFO|DFW|    4|    3|
|ABQ|SFO|ORD|    4|  124|
|ABQ|SFO|LAX|    4|  139|
|ABQ|SFO|JFK|    4|  133|
|ABQ|SFO|ORD|    4|  113|
|ABQ|SFO|LAX|    4|    8|
|ABQ|SFO|MIA|    4|   18|
|ABQ|SFO|DFW|    4|    2|
|ABQ|SFO|ORD|    4|    9|
|ABQ|SFO|ORD|    4|  326|
|ABQ|SFO|DFW|    4|    1|
|ABQ|SFO|ORD|    4|   34|
|ABQ|SFO|DFW|    4|    1|
|ABQ|SFO|ORD|    4|  190|
|ABQ|SFO|LAX|    4|    9|
|ABQ|SFO|JFK|    4|  111|
|ABQ|SFO|DFW|    4|  103|
+---+---+---+-----+-----+
only showing top 20 rows


### Determine Airport Ranking in Desc. order using PageRank algorithm

In [30]:
results = g.pageRank(
    resetProbability=0.15,
    maxIter=10
)
results.vertices.select("id","pagerank").orderBy("pagerank",ascending=False).show()

+---+------------------+
| id|          pagerank|
+---+------------------+
|ATL|29.615367151902028|
|DFW|21.412549106054627|
|ORD|20.784764668927355|
|DEN|15.525851214594773|
|LAX|14.240991985905305|
|IAH|12.566621320447185|
|SFO|11.258453926970866|
|PHX| 10.55374157361721|
|SLC| 9.330416999448307|
|LAS| 8.534780587472302|
|SEA| 7.370610900510238|
|MCO| 7.190221731843621|
|CLT| 7.181012806544953|
|EWR|7.1269623643872615|
|DTW| 6.845866070853457|
|LGA| 6.793762191654391|
|MSP|   6.7309186470033|
|BOS| 6.268854230795433|
|BWI| 5.731561010174128|
|JFK| 5.701883676390318|
+---+------------------+
only showing top 20 rows


## Determine the most popular flights (single city hops)

In [31]:
g.edges.groupBy("src","dst").agg(count("*").alias("numFlights")).orderBy("numFlights",ascending=False).show(20)

+---+---+----------+
|src|dst|numFlights|
+---+---+----------+
|SFO|LAX|      3232|
|LAX|SFO|      3198|
|LAS|LAX|      3016|
|LAX|LAS|      2964|
|JFK|LAX|      2720|
|LAX|JFK|      2719|
|ATL|LGA|      2501|
|LGA|ATL|      2500|
|LAX|PHX|      2394|
|PHX|LAX|      2387|
|HNL|OGG|      2380|
|OGG|HNL|      2379|
|LAX|SAN|      2215|
|SAN|LAX|      2214|
|SJC|LAX|      2208|
|LAX|SJC|      2201|
|ATL|MCO|      2136|
|MCO|ATL|      2090|
|JFK|SFO|      2084|
|SFO|JFK|      2084|
+---+---+----------+
only showing top 20 rows


### Find and Save a Subragph that obtained from the following pattern:
#### The flight starts from an airport and return back to the same airport through 2 other airports.

In [42]:
df_sub = g.find(
    "(a)-[ab]->(b); \
     (b)-[bc]->(c); \
     (c)-[ca]->(a)"
)

In [43]:
df_sub.printSchema()

root
 |-- a: struct (nullable = false)
 |    |-- City: string (nullable = true)
 |    |-- State: string (nullable = true)
 |    |-- Country: string (nullable = true)
 |    |-- id: string (nullable = true)
 |-- ab: struct (nullable = false)
 |    |-- tripid: integer (nullable = true)
 |    |-- delay: integer (nullable = true)
 |    |-- distance: integer (nullable = true)
 |    |-- src: string (nullable = true)
 |    |-- dst: string (nullable = true)
 |-- b: struct (nullable = false)
 |    |-- City: string (nullable = true)
 |    |-- State: string (nullable = true)
 |    |-- Country: string (nullable = true)
 |    |-- id: string (nullable = true)
 |-- bc: struct (nullable = false)
 |    |-- tripid: integer (nullable = true)
 |    |-- delay: integer (nullable = true)
 |    |-- distance: integer (nullable = true)
 |    |-- src: string (nullable = true)
 |    |-- dst: string (nullable = true)
 |-- c: struct (nullable = false)
 |    |-- City: string (nullable = true)
 |    |-- State: string 

In [44]:
e1 = df_sub.select(
    "ab.src",
    "ab.dst",
    "ab.tripid",
    "ab.delay",
    "ab.distance"
)

In [45]:
e2 = df_sub.select(
    "bc.src",
    "bc.dst",
    "bc.tripid",
    "bc.delay",
    "bc.distance"
)

In [46]:
e3 = df_sub.select(
    "ca.src",
    "ca.dst",
    "ca.tripid",
    "ca.delay",
    "ca.distance"
)
e = e1.union(e2).union(e3).distinct()

In [47]:
v1 = df_sub.select(
    "a.id",
    "a.City",
    "a.State",
    "a.Country"
)

In [48]:
v2 = df_sub.select(
    "b.id",
    "b.City",
    "b.State",
    "b.Country"
)

In [49]:
v3 = df_sub.select(
    "c.id",
    "c.City",
    "c.State",
    "c.Country"
)
v = v1.union(v2).union(v3).distinct()

In [50]:
gf_sub = GraphFrame(v, e)

In [51]:
gf_sub.vertices.write.mode("overwrite").parquet(
    "cycle_vertices"
)

gf_sub.edges.write.mode("overwrite").parquet(
    "cycle_edges"
)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 